<a href="https://colab.research.google.com/github/krishnakumar424w/AI-AGENTS-CLASS-1/blob/main/day5_assisgnment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U langchain langchain-google-genai langgraph


In [ ]:
import os
from google.colab import userdata

# Retrieve key from Colab secrets, or set os.environ["GEMINI_API_KEY"] = "YOUR_KEY"
api_key = userdata.get("GEMINI_API_KEY")
os.environ["GEMINI_API_KEY"] = api_key

In [ ]:
import sqlite3

def init_db():
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS students")
    cursor.execute("""
        CREATE TABLE students (
            student_id TEXT PRIMARY KEY,
            name TEXT,
            department TEXT,
            python INTEGER,
            database_score INTEGER,
            ai INTEGER,
            web INTEGER
        )
    """)

    students_data = [
        ('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78),
        ('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72),
        ('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90),
        ('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62),
        ('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88)
    ]

    cursor.executemany("""
        INSERT INTO students VALUES (?, ?, ?, ?, ?, ?, ?)
    """, students_data)

    conn.commit()
    conn.close()
    print("Database students.db created and populated successfully.")

init_db()

Database students.db created and populated successfully.


In [ ]:
import sqlite3
import ast
import operator
from langchain_core.tools import tool

@tool
def get_student_info(student_id: str) -> str:
    """Fetch the student's name and department using their student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name, department FROM students WHERE student_id = ?", (student_id.strip(),))
    row = cursor.fetchone()
    conn.close()

    if row:
        return f"Student ID: {student_id}, Name: {row[0]}, Department: {row[1]}"
    return f"Student with ID {student_id} not found."

@tool
def get_student_marks(student_id: str) -> str:
    """Fetch subject marks (Python, Database, AI, Web) for a given student ID."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT python, database_score, ai, web FROM students WHERE student_id = ?", (student_id.strip(),))
    row = cursor.fetchone()
    conn.close()

    if row:
        return f"Python: {row[0]}, Database: {row[1]}, AI: {row[2]}, Web: {row[3]}"
    return f"Marks for student ID {student_id} not found."

@tool
def calculator(expression: str) -> str:
    """Calculate math expressions safely to find total, average, percentages, etc.
    Input should be a mathematical expression like '(85 + 72 + 90 + 78)' or '325 / 4'.
    """
    ops = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Mod: operator.mod,
        ast.Pow: operator.pow
    }

    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        elif isinstance(node, ast.BinOp):
            return ops[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp):
            return -_eval(node.operand)
        raise ValueError("Unsupported operation")

    try:
        node = ast.parse(expression.strip(), mode='eval').body
        return str(_eval(node))
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

@tool
def get_passing_rules() -> str:
    """Returns the university passing criteria and minimum requirements."""
    return (
        "University Passing Rules:\n"
        "1. Minimum mark required in each subject: 35%\n"
        "2. Minimum overall average mark: 40%"
    )

tools = [get_student_info, get_student_marks, calculator, get_passing_rules]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

# Create an agent capable of dynamic multi-step reasoning
agent_executor = create_react_agent(llm, tools)

/tmp/ipykernel_3757/1597454023.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [ ]:
def run_query(query: str):
    print(f"\n{'='*20}\nQuestion: {query}\n{'='*20}")
    response = agent_executor.invoke({"messages": [("user", query)]})
    # Print final agent answer
    print("\nAnswer:\n", response["messages"][-1].content)

# Test 1: Single tool (Info)
run_query("What is the name and department of student 22CS045?")

# Test 2: Single tool (Marks)
run_query("What are the marks of 22CS047?")

# Test 3: Marks + Calculator
run_query("What is the total and average mark of 22CS045?")

# Test 4: Marks + Rules + Calculator
run_query("Is 22CS045 eligible to pass according to the university rules?")

# Challenge Question: Multi-tool chain
run_query(
    "I am 22CS045. Tell me my name, department, total marks, average marks, "
    "and whether I satisfy the university passing requirements."
)


Question: What is the name and department of student 22CS045?

Answer:
 [{'type': 'text', 'text': 'The student with ID **22CS045** is **Dhanushya** from the **Computer Science** department.', 'extras': {'signature': 'EvkBCvYBAWkUfRP7pcrMwCzT9/YkQh0yLHyHT+plzFsPAQnQoXQ73rKxXexHZkzpSr+gnlPfoPu6ZNYkN1Zx/I7+nKrIbjyZdK66aPbvp3LY+S7PryhPKPinPA4UEeDQRwqPabrSqWxl5NP6ymvTKi6qjSu9hUw55ezxxm4nnGsGp5Qvav0Z4qEdXiWazGg4xN9MTISmxFLIfqLdxfLb4LylNz6gRDfQQhaHlm3SY/prOKmuv500EfGcRdz30rC8Ei0QchGal9YAXD2k0nptfko78uBHoZ66Kl641wPs3k7Ch6haUCb/dp/ut0IE8WAcOKPkZ/5ukB6VXbp9'}}]

Question: What are the marks of 22CS047?


GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 55.379618995s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '